# 01 — V63 snapshot inputs with enrollment padding

This is a fresh **snapshot-only** experiment: all 49 already-encoded V63 `MODEL_DATA` values are matched to their exact `PATIENT_ID, END_DT` snapshot and repeated unchanged at enrollment-valid positions. There is **no reconstruction of historical feature values and no monthly-to-quarterly feature aggregation**. Monthly and quarterly inputs differ in the enrollment mask and number of positions, not in the underlying snapshot feature vector.

Enrollment validity means at least one inclusive date overlap with **any** enrollment interval. It does not require a full month of coverage or `MX_CLOSED`/`RX_CLOSED`; it does not establish feature-specific observability. Missing feature values remain NaN at valid positions for notebook 02's TRAIN-only imputation. Invalid positions are numeric zero with a separate Boolean mask. Valid zeros remain valid.

Run notebooks **01 → 02 → 03 → 04** with the same fresh IDs. Source cohort and frozen patient assignments are retained; no earlier tensor, trained model, prediction or experiment result is consumed. This notebook reads source tables and saves only to the fresh private output namespace. It does not train models. Notebook code contains no credentials; supply an approved private connection in your Databricks environment.

**Interpretation:** repeating a snapshot does not create a time trend. Any monthly/quarterly difference cannot be described as learning different historical feature trajectories. Snapshot timing and upstream V63 encoding/feature selection require their own leakage review before a prospective claim. The checks below establish extraction, alignment and padding integrity, not causal validity or deployment readiness.


## Step 1 — Connection and explicit experiment contract

**Why:** Give all four notebooks one reproducible input contract and a new output namespace.

**Next cell output:** Dataset ID, source vintage, design description and destination table; connection secrets are never printed.

**How to interpret it:** These IDs distinguish this rebuilt experiment. Source vintage is fixed. Change DATASET_ID across all four notebooks if the input policy or source data changes.


In [ ]:
import os
import re
import io
import json
import math
import base64
import hashlib
import numpy as np
import pandas as pd

if 'spark' not in globals() or 'sf_options' not in globals() or not isinstance(sf_options, dict) or not sf_options:
    raise RuntimeError('Supply spark and sf_options securely in the approved Databricks runtime. Never paste or print credentials.')
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
SCHEMA = 'DS_ML'
sf_options_dl_poc = dict(sf_options)
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema=SCHEMA)
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
SOURCE_VINTAGE = '20260825'
DATASET_ID = 'D20260927S'
RUN_ID = 'R20260927S'
for identifier in (DATASET_ID, RUN_ID):
    if re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', identifier) is None:
        raise ValueError('Use short uppercase dataset/run IDs, identical in all four notebooks.')
EXPERIMENT_PREFIX = SOURCE_PREFIX + '_DL_POC_SNAPSHOT_PAD_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SOURCE_SNAPSHOT_TABLE = PREFIX + '_SNAPSHOTS'
SOURCE_SPLIT_TABLE = PREFIX + '_PATIENT_SPLIT'
ENROLLMENT_TABLE = 'ENROLLMENT_' + SOURCE_VINTAGE
SNAPSHOTS_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_SNAPSHOT_TABLE
SPLIT_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_SPLIT_TABLE
ENROLLMENT_FQN = DATABASE + '.' + SCHEMA + '.' + ENROLLMENT_TABLE
MODEL_DATA_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_PREFIX + '_MODEL_DATA'
MODEL_TYPE_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_PREFIX + '_MODEL_TYPE'
FINAL_MODEL_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_PREFIX + '_FINAL_MODEL'
FEATURES_SUMMARY_FQN = DATABASE + '.' + SCHEMA + '.' + SOURCE_PREFIX + '_FEATURES_SUMMARY'
PREPARED_TABLE_FQN = DATABASE + '.' + SCHEMA + '.' + PREPARED_TABLE
EXPECTED_POPULATION = {'snapshots': 23151, 'patients': 12447, 'positives': 1345}
EXPECTED_SPLIT = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
FEATURE_POLICY = 'v63_snapshot_only_valid_timestep_broadcast_v1'
INPUT_DESCRIPTION = 'V63 snapshot features repeated only at enrollment-valid timesteps; no historical feature reconstruction'
PADDING_CONTRACT = 'independent_timestep_availability_v1'
ORIENTATION = '0=newest; snapshot-relative month buckets; newest ends at END_DT'
BUCKET_DEFINITION = {
    'MONTHLY': {'steps': 12, 'months_per_step': 1},
    'QUARTERLY': {'steps': 4, 'months_per_step': 3},
    'anchor': 'snapshot calendar month',
    'start': 'first day of anchor month minus width*step minus (width-1) months',
    'end': 'min(END_DT, last day of anchor month minus width*step months)'}
COVERAGE_RULE_TEXT = 'e.PATIENT_ID = g.PATIENT_ID AND e.START_DATE <= g.PERIOD_END AND e.END_DATE >= g.PERIOD_START'
COVERAGE_POLICY_TEXT = 'Any enrollment interval with inclusive date overlap; at least one covered day is sufficient; no MX_CLOSED/RX_CLOSED restriction.'
IMPLEMENTATION_SHA256 = 'c3891483bbd0968cdcb0a7f288f631d6b30aa7346901b42250a0f6bbc4c55316'
ARTIFACT_COLUMNS = ['ARTIFACT_NAME', 'CHUNK_INDEX', 'CHUNK_COUNT', 'BYTE_LENGTH', 'SHA256', 'PAYLOAD_BASE64']
print('Dataset:', DATASET_ID, '| source vintage:', SOURCE_VINTAGE)
print('Design:', INPUT_DESCRIPTION)
print('Private output:', PREPARED_TABLE_FQN)


## Step 2 — Source access and verified storage helpers

**Why:** Keep private reads and hashed exports in this notebook, without external helper files.

**Next cell output:** Confirmation that helper functions are defined; no source rows are read yet.

**How to interpret it:** Saved payloads use exact byte hashes and read-back verification. An existing destination with different contents is rejected rather than overwritten.


In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()


def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new DATASET_ID for changed prepared inputs.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")

def read_query(query, options=None):
    return (spark.read.format('snowflake').options(**(sf_options_dl_poc if options is None else options))
            .option('query', query).load())

write_artifacts = save_artifacts
print('Private source readers and non-overwriting artifact storage are defined.')


## Step 3 — Authoritative feature names and encoding audit

**Why:** Use the actual ordered 49-feature contract instead of a manually transcribed list. Confirm MODEL_TYPE.FEATURES matches FINAL_MODEL ordered by SEQ.

**Next cell output:** Helper definitions. The feature list and frozen encoding records are read in step 6.

**How to interpret it:** VALUE_P and VAR_TYP are audited only. MODEL_DATA is already encoded, so caps, binary conversion and AGE encoding are not applied again. These frozen upstream parameters were not fitted in this notebook.


In [ ]:
def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() or x != x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value


def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(0 < len(result) <= 49, 'Retained predictors must be a nonempty subset of the 49 candidates.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == len(result),
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (selected ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    selected feature list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)


def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'feature_role': 'All 49 frozen predictors retained, with exact source order'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'already encoded V63 MODEL_DATA snapshot values', 'encoding_applied_here': False, 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract

print('Feature-order reconciliation and fixed V63 encoding audit helpers are defined.')


## Step 4 — Strict row, date, label and value validation

**Why:** Prevent silent conversion, reordered tensors or patient leakage from creating misleading results.

**Next cell output:** Helper definitions for exact source alignment and type checks.

**How to interpret it:** Bad labels, malformed feature values, extra/missing snapshot keys, time-of-day cutoffs and split leakage stop execution. Genuine source NaNs remain missing; text is never silently converted into NaN.


In [ ]:
def exact_dates(values, name):
    require(not values.isna().any(), name + ': missing dates are not allowed.')
    require(not values.map(lambda v: isinstance(v, (bool, np.bool_, int, float, np.number))).any(),
            name + ': numeric/Boolean values cannot be interpreted as dates.')
    dates = pd.to_datetime(values, errors='raise')
    require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(), name + ': expected exact dates without timezone/time of day.')
    return dates.dt.strftime('%Y-%m-%d')


def normalize_metadata(frame):
    require(not frame.columns.duplicated().any(), 'Duplicate metadata columns.')
    out = frame[['PATIENT_ID', 'END_DT', 'RESP']].copy().reset_index(drop=True)
    require(not out.empty and not out.isna().any().any(), 'Missing snapshot keys or labels.')
    require(out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x) and x == x.strip()).all(),
            'Patient IDs must remain exact nonempty strings without surrounding whitespace.')
    out['END_DT'] = exact_dates(out.END_DT, 'END_DT')
    require(not out.RESP.map(lambda x: isinstance(x, (str, bool, np.bool_))).any()
            and out.RESP.isin([0, 1]).all(), 'RESP must be numeric 0 or 1, without coercion.')
    out['RESP'] = out.RESP.astype('int64')
    require(not out.duplicated(['PATIENT_ID', 'END_DT']).any(), 'Duplicate patient/date keys; no automatic deduplication.')
    return out


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def strict_numeric_features(frame, features):
    require(not frame.columns.duplicated().any(), 'Duplicate model-data columns.')
    require(not any(pd.api.types.is_datetime64_any_dtype(frame[name]) or pd.api.types.is_timedelta64_dtype(frame[name]) for name in features),
            'MODEL_DATA predictors must be numeric encoded values, not date/time columns.')
    converted = frame[features].apply(pd.to_numeric, errors='raise')
    require(not any(np.iscomplexobj(converted[name].to_numpy()) for name in features),
            'MODEL_DATA predictors must be real numeric values, not complex numbers.')
    require(not (converted.isna() & ~frame[features].isna()).any().any(),
            'A nonmissing source value was converted to missing; fix the source.')
    values = converted.to_numpy(dtype=np.float32)
    require(not np.isinf(values).any(), 'MODEL_DATA contains infinite values or overflows float32.')
    return pd.DataFrame(values, columns=features, index=frame.index)


def align_snapshot_values(metadata, model_data, features):
    original = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    data = model_data.reset_index(drop=True).copy()
    data[['PATIENT_ID', 'END_DT', 'RESP']] = normalize_metadata(data)
    require(len(data) == len(original), 'MODEL_DATA and frozen population have different snapshot counts.')
    merged = original.assign(_ROW_ORDER=np.arange(len(original))).merge(
        data, on=['PATIENT_ID', 'END_DT'], how='outer', suffixes=('_SNAP', '_MODEL'),
        indicator=True, validate='one_to_one', sort=False)
    require(merged['_merge'].eq('both').all(), 'MODEL_DATA keys differ from the exact frozen population.')
    merged = merged.sort_values('_ROW_ORDER').reset_index(drop=True)
    require(np.array_equal(merged.RESP_SNAP.to_numpy(), merged.RESP_MODEL.to_numpy()), 'RESP differs between SNAPSHOTS and MODEL_DATA.')
    aligned_metadata = merged[['PATIENT_ID', 'END_DT', 'RESP_SNAP']].rename(columns={'RESP_SNAP': 'RESP'})
    require(original.equals(aligned_metadata), 'Source alignment changed population row order.')
    values = strict_numeric_features(merged, features)
    return aligned_metadata, values


def validate_frozen_split(metadata, frozen, expected=None):
    require('SPLIT' in frozen.columns, 'Frozen source split is missing SPLIT.')
    labels = frozen['SPLIT'].reset_index(drop=True)
    require(labels.map(lambda v: isinstance(v, str) and v in {'train', 'validation', 'test'}).all(), 'Unexpected/null frozen split labels.')
    normalized = normalize_metadata(frozen)
    normalized['SPLIT'] = labels.to_numpy()
    normalized = normalized.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    reference = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(reference.equals(normalized[['PATIENT_ID', 'END_DT', 'RESP']]), 'Frozen split keys/labels differ from the exact source population.')
    require(set(normalized.SPLIT) == {'train', 'validation', 'test'}, 'All three frozen split labels are required.')
    require(normalized.groupby('PATIENT_ID').SPLIT.nunique().eq(1).all(), 'A patient appears in more than one split.')
    statistics = []
    for name in ('train', 'validation', 'test'):
        part = normalized.loc[normalized.SPLIT.eq(name)]
        observed = (len(part), int(part.PATIENT_ID.nunique()), int(part.RESP.sum()))
        if expected is not None:
            require(observed == expected[name], name + ': frozen split counts changed.')
        statistics.append({'SPLIT': name, 'SNAPSHOTS': observed[0], 'PATIENTS': observed[1], 'POSITIVES': observed[2]})
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    return normalized, pd.DataFrame(statistics), digest_json(records)


def strict_booleans(values, name):
    require(not values.isna().any(), name + ': missing Boolean values.')
    def parse(value):
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if isinstance(value, (int, np.integer)) and value in (0, 1):
            return bool(value)
        if isinstance(value, str) and value in ('TRUE', 'FALSE'):
            return value == 'TRUE'
        raise ValueError(name + ': expected bool, integer 0/1 or exact TRUE/FALSE strings; no truthiness coercion.')
    return values.map(parse).astype(bool)

print('Strict source types, exact row alignment and patient-disjoint split checks are defined.')


## Step 5 — Calendar and enrollment mask definitions

**Why:** Create 12 monthly positions or 4 corresponding three-month positions, with position 0 newest and the newest period ending at snapshot END_DT.

**Next cell output:** Calendar, SQL and mask-alignment helper definitions.

**How to interpret it:** A true mask means any interval overlaps the period, even one day. Quarterly validity must equal OR across each matching group of three monthly masks. This is a mask aggregation only; feature values are not aggregated.


In [ ]:
def sequence_grid(metadata, representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    rows = []
    for row in metadata.itertuples():
        cutoff = pd.Timestamp(row.END_DT)
        month = cutoff.to_period('M')
        for step in range(12 // width):
            start = (month - width * step - (width - 1)).start_time.normalize()
            end = min(cutoff, (month - width * step).end_time.normalize())
            rows.append((row.PATIENT_ID, row.END_DT, step, start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d'), int(row.RESP)))
    grid = pd.DataFrame(rows, columns=['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'RESP'])
    require(not grid.duplicated(['PATIENT_ID', 'END_DT', 'TIME_STEP']).any(), 'Duplicate sequence keys.')
    require(grid.PERIOD_START.le(grid.PERIOD_END).all() and grid.PERIOD_END.le(grid.END_DT).all(), 'Invalid/future period boundary.')
    return grid


def verify_periods(monthly, quarterly):
    keys = ['PATIENT_ID', 'END_DT']
    for frame, count in ((monthly, 12), (quarterly, 4)):
        require(not frame.duplicated(keys + ['TIME_STEP']).any(), 'Duplicate sequence position.')
        require(frame.groupby(keys).TIME_STEP.apply(lambda x: sorted(x) == list(range(count))).all(), 'Invalid sequence positions.')
    for step in range(4):
        monthly_triple = monthly.loc[monthly.TIME_STEP.between(3*step, 3*step+2)]
        expected = monthly_triple.groupby(keys).agg(PERIOD_START=('PERIOD_START', 'min'), PERIOD_END=('PERIOD_END', 'max')).sort_index()
        actual = quarterly.loc[quarterly.TIME_STEP.eq(step)].set_index(keys)[['PERIOD_START', 'PERIOD_END']].sort_index()
        require(expected.equals(actual), 'Quarterly calendar is not the union of matching monthly triples.')


def coverage_query(representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    steps = 12 // width
    return f"""
    WITH grid AS (
        SELECT s.PATIENT_ID, s.END_DT, seq.STEP AS TIME_STEP,
               DATEADD('month', -({width} * seq.STEP + ({width} - 1)), DATE_TRUNC('month', s.END_DT)) AS PERIOD_START,
               LEAST(s.END_DT, LAST_DAY(DATEADD('month', -({width} * seq.STEP), DATE_TRUNC('month', s.END_DT)))) AS PERIOD_END
        FROM {SNAPSHOTS_FQN} s
        CROSS JOIN (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS STEP
                    FROM TABLE(GENERATOR(ROWCOUNT => {steps}))) seq
    )
    SELECT g.PATIENT_ID, g.END_DT, g.TIME_STEP, g.PERIOD_START, g.PERIOD_END,
           IFF(COUNT(e.PATIENT_ID) > 0, TRUE, FALSE) AS IS_VALID_TIMESTEP
    FROM grid g LEFT JOIN {ENROLLMENT_FQN} e
      ON e.PATIENT_ID = g.PATIENT_ID
     AND e.START_DATE <= g.PERIOD_END
     AND e.END_DATE >= g.PERIOD_START
    GROUP BY 1,2,3,4,5
    ORDER BY 1,2,3
    """


def align_coverage(grid, coverage, representation, population_size):
    columns = ['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'IS_VALID_TIMESTEP']
    require(not coverage.columns.duplicated().any() and set(columns).issubset(coverage), 'Incomplete/duplicate coverage columns.')
    coverage = coverage[columns].copy().reset_index(drop=True)
    require(coverage.PATIENT_ID.map(lambda v: isinstance(v, str) and bool(v) and v == v.strip()).all(), 'Invalid coverage patient keys.')
    for name in ('END_DT', 'PERIOD_START', 'PERIOD_END'):
        coverage[name] = exact_dates(coverage[name], name)
    step = coverage.TIME_STEP
    require(not step.isna().any() and not step.map(lambda x: isinstance(x, (str, bool, np.bool_))).any(), 'Invalid coverage timestep types.')
    require(np.isfinite(step).all() and step.eq(np.floor(step)).all(), 'Coverage timesteps must be finite integer ranks.')
    coverage['TIME_STEP'] = step.astype('int64')
    coverage['IS_VALID_TIMESTEP'] = strict_booleans(coverage.IS_VALID_TIMESTEP, 'IS_VALID_TIMESTEP')
    keys = columns[:5]
    require(not coverage.duplicated(keys[:3]).any(), 'Duplicate coverage snapshot/timestep rows.')
    require(len(coverage) == len(grid), 'Coverage row count differs from calendar grid.')
    merged = grid.assign(_ROW_ORDER=np.arange(len(grid))).merge(coverage, on=keys, how='outer', sort=False, validate='one_to_one', indicator=True)
    require(merged['_merge'].eq('both').all(), 'Coverage keys/boundaries do not match the calendar grid exactly.')
    merged = merged.sort_values('_ROW_ORDER').drop(columns=['_merge', '_ROW_ORDER']).reset_index(drop=True)
    require(merged[grid.columns].equals(grid.reset_index(drop=True)), 'Coverage alignment changed calendar order.')
    steps = 12 if representation == 'MONTHLY' else 4
    valid = merged.IS_VALID_TIMESTEP.to_numpy(dtype=bool).reshape(population_size, steps)
    return merged, valid


def load_coverage(representation, grid):
    coverage = read_query(coverage_query(representation)).toPandas()
    merged, valid = align_coverage(grid, coverage, representation, len(metadata))
    audit = {
        'available_timesteps': int(valid.sum()), 'unavailable_timesteps': int((~valid).sum()),
        'unknown_timesteps': 0, 'all_padded_snapshots': int((~valid.any(axis=1)).sum()),
        'policy': COVERAGE_POLICY_TEXT,
        'evidence': ENROLLMENT_FQN + ' with [' + COVERAGE_RULE_TEXT + '] on the ' + representation.lower() + ' snapshot calendar grid.',
        'full_period_coverage_required': False, 'closed_claims_filter_applied': False}
    return merged, valid, audit

print('Monthly/quarterly calendar and independent enrollment-mask helpers are defined.')


## Step 6 — Load and reconcile the fixed source snapshots

**Why:** Bind every 49-feature vector and label to the same exact patient/date row and retain the original patient-level assignments.

**Next cell output:** Aggregate cohort/split counts and the exact ordered feature list with frozen types. No patient rows are displayed.

**How to interpret it:** Expect 23,151 snapshots, 12,447 patients, 1,345 positives and 49 snapshot features. Temporal features must be zero by this requested design. Missing values here are individual feature gaps, not a reason to pad an otherwise valid timestep.


In [ ]:
features, configuration_audit = configured_features()
business_parameters, feature_summary, encoding_contract = load_business_parameters(features)
metadata = normalize_metadata(read_table(SOURCE_SNAPSHOT_TABLE).select('PATIENT_ID', 'END_DT', 'RESP').toPandas())
metadata = metadata.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
require((len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum())) ==
        (EXPECTED_POPULATION['snapshots'], EXPECTED_POPULATION['patients'], EXPECTED_POPULATION['positives']), 'Frozen population counts changed.')
model_data = read_table(SOURCE_PREFIX + '_MODEL_DATA').select(['PATIENT_ID', 'END_DT', 'RESP'] + features).toPandas()
metadata, source_snapshot_values = align_snapshot_values(metadata, model_data, features)
frozen_source = read_table(SOURCE_SPLIT_TABLE).select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT').toPandas()
frozen_split, source_split_summary, source_split_sha256 = validate_frozen_split(metadata, frozen_source, EXPECTED_SPLIT)
snapshot_matrix = source_snapshot_values.to_numpy(dtype=np.float32)
source_summary = pd.DataFrame([
    {'ITEM': 'population_snapshots', 'VALUE': len(metadata)},
    {'ITEM': 'population_patients', 'VALUE': metadata.PATIENT_ID.nunique()},
    {'ITEM': 'population_positives', 'VALUE': int(metadata.RESP.sum())},
    {'ITEM': 'feature_count', 'VALUE': len(features)},
    {'ITEM': 'temporal_features', 'VALUE': 0},
    {'ITEM': 'snapshot_features', 'VALUE': len(features)},
    {'ITEM': 'missing_snapshot_feature_values', 'VALUE': int(np.isnan(snapshot_matrix).sum())}])
display(source_summary)
display(source_split_summary)
display(pd.DataFrame(business_parameters)[['FEATURE_ORDER', 'FEATURES', 'VAR_TYP', 'TRANSFORM']])
print('Exact population, source values, target labels, feature order and patient-disjoint frozen split verified.')


## Step 7 — Validate enrollment and build masks

**Why:** Use dates in the enrollment source independently of feature values to decide whether each period is included or padded.

**Next cell output:** Enrollment interval quality counts and monthly/quarterly available, unavailable and all-padded counts.

**How to interpret it:** Null/reversed cohort intervals are rejected. Some snapshots may be entirely padded when they have no overlap. A lower quarterly padding rate is expected because one covered month makes the three-month period valid; it does not prove better data quality or predictive performance. Overlapping intervals do not double-count validity because the rule is existence.


In [ ]:
enrollment_schema = {field.name.upper(): field.dataType.simpleString().lower() for field in read_table(ENROLLMENT_TABLE).schema.fields}
require(set(['PATIENT_ID', 'START_DATE', 'END_DATE']).issubset(enrollment_schema), 'Enrollment source must expose PATIENT_ID, START_DATE, END_DATE.')
require(enrollment_schema['START_DATE'] == 'date' and enrollment_schema['END_DATE'] == 'date',
        'Enrollment boundaries must be DATE columns; inspect any timestamp/string source before use.')
enrollment_quality = read_query(f"""
    WITH cohort AS (SELECT DISTINCT PATIENT_ID FROM {SNAPSHOTS_FQN}),
    relevant AS (SELECT e.PATIENT_ID, e.START_DATE, e.END_DATE FROM {ENROLLMENT_FQN} e
                 INNER JOIN cohort c ON c.PATIENT_ID = e.PATIENT_ID)
    SELECT COUNT(*) AS ENROLLMENT_ROWS,
           COALESCE(COUNT_IF(START_DATE IS NULL), 0) AS NULL_START_DATE_ROWS,
           COALESCE(COUNT_IF(END_DATE IS NULL), 0) AS NULL_END_DATE_ROWS,
           COALESCE(COUNT_IF(START_DATE > END_DATE), 0) AS REVERSED_INTERVAL_ROWS
    FROM relevant
""").toPandas()
require(len(enrollment_quality) == 1, 'Unexpected enrollment quality report shape.')
quality_row = enrollment_quality.iloc[0]
for column in ['NULL_START_DATE_ROWS', 'NULL_END_DATE_ROWS', 'REVERSED_INTERVAL_ROWS']:
    require(int(quality_row[column]) == 0, 'Enrollment source has invalid intervals: ' + column)
enrollment_quality_record = {name: int(quality_row[name]) for name in enrollment_quality.columns}
monthly_grid = sequence_grid(metadata, 'MONTHLY')
quarterly_grid = sequence_grid(metadata, 'QUARTERLY')
verify_periods(monthly_grid, quarterly_grid)
monthly_long, monthly_valid, monthly_coverage_audit = load_coverage('MONTHLY', monthly_grid)
quarterly_long, quarterly_valid, quarterly_coverage_audit = load_coverage('QUARTERLY', quarterly_grid)
require(np.array_equal(quarterly_valid, monthly_valid.reshape(len(metadata), 4, 3).any(axis=2)),
        'Quarterly validity must equal the OR of matching monthly triples.')
require(np.array_equal(monthly_valid.any(axis=1), quarterly_valid.any(axis=1)), 'All-padded snapshot status differs by representation.')
display(enrollment_quality)
display(pd.DataFrame([{'MODEL': 'MONTHLY', **monthly_coverage_audit}, {'MODEL': 'QUARTERLY', **quarterly_coverage_audit}]))
print('Coverage predicate:', COVERAGE_RULE_TEXT)
print('Quarterly OR-of-three-month mask and identical all-padded snapshot status verified.')


## Step 8 — Broadcast the snapshot vector and verify padding

**Why:** Implement the requested fallback consistently for all 49 features, while preventing invalid periods from becoming model inputs.

**Next cell output:** Tensor dimensions, missing-value counts, padding counts and feature-source audit.

**How to interpret it:** Expected shapes are [23,151, 12, 49] and [23,151, 4, 49]. Within each snapshot all valid feature vectors are identical. Repetition is deliberate snapshot context, not recovered history. All-padded rows have no feature information and must be handled explicitly downstream. A nonzero snapshot value never determines validity.


In [ ]:
fallback_reason = 'Requested snapshot-only rebuild: exact already-encoded V63 MODEL_DATA values are matched by PATIENT_ID and END_DT, then repeated at valid positions only; no feature histories are reconstructed.'
validation_text = 'Validated unique exact source keys, labels and row order, numeric types, finite-or-missing encoded values, valid-only constant broadcast, cross-representation agreement, zero padding, exact masks and artifact hashes.'
audit_rows = []
for name in features:
    audit_rows.append({
        'FEATURE_NAME': name, 'STATUS': 'IMPLEMENTED_VALIDATED', 'VALUE_SOURCE': 'V63_SNAPSHOT',
        'SOURCE': MODEL_DATA_FQN,
        'CALCULATION': 'Read the already-encoded V63 snapshot value from MODEL_DATA and repeat it unchanged across valid timesteps only.',
        'LOOKBACK': 'Snapshot END_DT context only; no historical lookback reconstructed.',
        'COVERAGE_RULE': COVERAGE_RULE_TEXT,
        'TEMPORAL_JUSTIFICATION': 'Snapshot context retained intentionally; this feature has no reconstructed temporal trajectory.',
        'VALIDATION': validation_text, 'SNAPSHOT_JOIN': ['PATIENT_ID', 'END_DT'],
        'ALREADY_ENCODED': True, 'BROADCAST': 'valid_timesteps_only', 'FALLBACK_REASON': fallback_reason})


def build_bundle(representation, values, valid, coverage_audit):
    steps = 12 if representation == 'MONTHLY' else 4
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    values = np.asarray(values)
    require(values.ndim == 2 and values.dtype == np.float32 and not np.isinf(values).any(), 'Invalid snapshot matrix.')
    require(valid.dtype == bool and valid.shape == (len(values), steps), 'Invalid validity-mask shape/type.')
    X = np.broadcast_to(values[:, None, :], (len(values), steps, values.shape[1])).copy()
    X[~valid] = 0.0
    require(X.dtype == np.float32 and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0), 'Padding must be finite numeric zero.')
    expected = np.broadcast_to(values[:, None, :], X.shape)
    require(np.array_equal(X[valid], expected[valid], equal_nan=True), 'Snapshot values changed at valid positions.')
    provenance = [
        {'step': 'feature_values', 'source': MODEL_DATA_FQN, 'join': ['PATIENT_ID', 'END_DT'],
         'value_source': 'V63_SNAPSHOT', 'already_encoded': True, 'encoding_applied_here': False,
         'broadcast': 'valid_timesteps_only', 'fallback_reason': fallback_reason},
        {'step': 'timestep_validity', 'source': ENROLLMENT_FQN, 'predicate': COVERAGE_RULE_TEXT,
         'policy': COVERAGE_POLICY_TEXT, 'quarterly_mask_rule': 'OR of matching monthly triples'}]
    return {'representation': representation, 'X': X, 'valid': valid.copy(),
            'coverage_audit': coverage_audit, 'availability_sha256': digest_json(coverage_audit),
            'provenance': provenance, 'shape': list(X.shape), 'value_space': 'V63 encoded',
            'X_sha256': array_hash(X), 'valid_sha256': array_hash(valid)}


monthly_bundle = build_bundle('MONTHLY', snapshot_matrix, monthly_valid, monthly_coverage_audit)
quarterly_bundle = build_bundle('QUARTERLY', snapshot_matrix, quarterly_valid, quarterly_coverage_audit)
bundles = {'MONTHLY': monthly_bundle, 'QUARTERLY': quarterly_bundle}
rows = np.flatnonzero(monthly_valid.any(axis=1) & quarterly_valid.any(axis=1))
if len(rows):
    a = monthly_bundle['X'][rows, monthly_valid[rows].argmax(axis=1), :]
    b = quarterly_bundle['X'][rows, quarterly_valid[rows].argmax(axis=1), :]
    require(np.array_equal(a, b, equal_nan=True), 'Snapshot values differ across representations.')
summary = []
for name, bundle in bundles.items():
    valid = bundle['valid']
    require(bundle['shape'] == [len(metadata), 12 if name == 'MONTHLY' else 4, len(features)], 'Unexpected tensor shape.')
    require(bundle['coverage_audit']['available_timesteps'] == int(valid.sum())
            and bundle['coverage_audit']['unavailable_timesteps'] == int((~valid).sum())
            and bundle['coverage_audit']['all_padded_snapshots'] == int((~valid.any(axis=1)).sum()), 'Coverage audit disagrees with mask.')
    summary.append({'MODEL': name, 'SNAPSHOTS': len(metadata), 'TIMESTEPS': valid.shape[1], 'FEATURES': len(features),
                    'SNAPSHOTS_WITH_ANY_VALID_STEP': int(valid.any(axis=1).sum()),
                    'PADDED_TIMESTEPS': int((~valid).sum()), 'MISSING_VALUES_AT_VALID_STEPS': int(np.isnan(bundle['X'][valid]).sum()),
                    'FEATURE_VALUES_VARY_WITHIN_VALID_HISTORY': False})
display(pd.DataFrame(summary))
display(pd.DataFrame(audit_rows)[['FEATURE_NAME', 'VALUE_SOURCE', 'SOURCE']])
print('Snapshot-only tensors built; valid zeros retained and padding separated by Boolean masks.')


## Step 9 — Save this input set and verify it after reading back

**Why:** Freeze the exact source-aligned values, masks, population, source contract and provenance for downstream processing.

**Next cell output:** Private artifact destination and the exact EXPECTED_PREPARED_MANIFEST_SHA256 value to paste into notebooks 02–04.

**How to interpret it:** A successful result confirms byte-level and array-level read-back checks, not model quality. The hash pins this input set so later notebooks cannot silently consume a changed dataset. No tensor values or patient identifiers are printed. Different contents at the same destination require a new DATASET_ID.


In [ ]:
population_json = canonical_json(snapshot_records(metadata)).encode('utf-8')
manifest = {
    'schema': 5, 'dataset_id': DATASET_ID, 'source_vintage': SOURCE_VINTAGE,
    'source_features': features, 'features': features, 'excluded_features': [], 'audit': audit_rows,
    'feature_policy': FEATURE_POLICY, 'input_description': INPUT_DESCRIPTION,
    'business_encoding': encoding_contract, 'encoding_applied_here': False,
    'padding_contract': PADDING_CONTRACT, 'orientation': ORIENTATION, 'bucket_definition': BUCKET_DEFINITION,
    'implementation_sha256': IMPLEMENTATION_SHA256,
    'population_sha256': digest_json(snapshot_records(metadata)),
    'snapshot_feature_sha256': array_hash(snapshot_matrix),
    'frozen_split_assignments_sha256': source_split_sha256,
    'preprocessing_policy': 'one_vector_per_eligible_train_snapshot_shared_v1',
    'configuration_audit': configuration_audit, 'enrollment_quality': enrollment_quality_record,
    'quarterly_feature_aggregation': 'None: same snapshot values; only enrollment masks change by bucket width.',
    'quarterly_mask_rule': 'OR of matching monthly triples',
    'source_tables': {'snapshots': SNAPSHOTS_FQN, 'model_data': MODEL_DATA_FQN, 'model_type': MODEL_TYPE_FQN,
                      'final_model': FINAL_MODEL_FQN, 'features_summary': FEATURES_SUMMARY_FQN,
                      'enrollment': ENROLLMENT_FQN, 'frozen_split': SPLIT_FQN},
    'representations': {name: {key: bundle[key] for key in
        ['shape', 'X_sha256', 'valid_sha256', 'coverage_audit', 'availability_sha256', 'provenance', 'value_space']}
        for name, bundle in bundles.items()}}
blobs = {'population.json': population_json}
for name, bundle in bundles.items():
    buffer = io.BytesIO()
    np.savez_compressed(buffer, X=bundle['X'], valid=bundle['valid'])
    blobs[name + '.npz'] = buffer.getvalue()
blobs['manifest.json'] = canonical_json(manifest).encode('utf-8')
write_artifacts(PREPARED_TABLE, blobs)
roundtrip = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
require(roundtrip == blobs, 'Read-back artifacts differ byte-for-byte.')
require(json.loads(roundtrip['manifest.json']) == manifest, 'Read-back manifest content differs.')
require(json.loads(roundtrip['population.json']) == snapshot_records(metadata), 'Read-back population order differs.')
for name, expected in bundles.items():
    with np.load(io.BytesIO(roundtrip[name + '.npz']), allow_pickle=False) as saved:
        require(set(saved.files) == {'X', 'valid'}, 'Unexpected saved arrays.')
        require(saved['X'].dtype == np.float32 and saved['valid'].dtype == bool, 'Saved array types changed.')
        require(np.array_equal(saved['X'], expected['X'], equal_nan=True)
                and np.array_equal(saved['valid'], expected['valid']), 'Saved tensor or mask changed.')
        require(array_hash(saved['X']) == manifest['representations'][name]['X_sha256']
                and array_hash(saved['valid']) == manifest['representations'][name]['valid_sha256'], 'Saved array hashes differ.')
manifest_sha256 = digest_json(manifest)
display(pd.DataFrame([{'VALUE_SOURCE': 'V63_SNAPSHOT', 'FEATURE_COUNT': len(features)},
                      {'VALUE_SOURCE': 'TEMPORAL', 'FEATURE_COUNT': 0}]))
print('Saved and verified prepared artifacts:', PREPARED_TABLE_FQN)
print('EXPECTED_PREPARED_MANIFEST_SHA256 =', repr(manifest_sha256))
print('Paste this exact hash into notebooks 02, 03 and 04, retain dataset/run IDs, then run notebook 02.')
